In [1]:
"""
=============================================================
Phase 5 — Block A: Executive Summary
=============================================================
Goal: Produce a concise, non-technical executive summary that
      tells the complete story of the project in plain English.

Audience: educators, institution leaders, HR/consulting teams,
          and recruiters — people who will NOT read the code.

Structure:
  1. Problem statement
  2. Data and methodology overview
  3. Key findings (bullet-pointed, plain English)
  4. Model performance summary
  5. Risk scoring output
  6. Recommendations preview
  7. Limitations and next steps

Output:
  • Printed to console
  • Saved as phase5_executive_summary.txt
"""

import os
import pickle
import warnings
warnings.filterwarnings('ignore')

from datetime import date
import pandas as pd
import numpy as np

# ── Config ────────────────────────────────────────────────────
BLOCK_A_PATH = './outputs/phase4/block_a/'
BLOCK_D_PATH = './outputs/phase4/block_d/'
BLOCK_E_PATH = './outputs/phase4/block_e/'
PHASE3_PATH  = './outputs/phase3/block_c/'
DATA_PATH    = './data/'
OUTPUT_PATH  = './outputs/phase5/block_a/'
os.makedirs(OUTPUT_PATH, exist_ok=True)

# ── Load key metrics ──────────────────────────────────────────
print("Loading metrics from all phases...")

si = pd.read_csv(DATA_PATH + 'studentInfo.csv')
si['dropout'] = (si['final_result'] == 'Withdrawn').astype(int)

# Phase 4 model metrics
with open(os.path.join(BLOCK_D_PATH, 'block_d_objects.pkl'), 'rb') as f:
    d_obj = pickle.load(f)

metrics_df      = d_obj['metrics_df']
best_model_name = d_obj['best_model_name']
best_threshold  = d_obj['best_threshold']
FEATURES        = d_obj['feature_names']

# Phase 4 risk scores
risk_table = pd.read_csv(
    os.path.join(BLOCK_E_PATH, 'E3_student_risk_scores.csv'),
    index_col=0
)

# Phase 3 logistic regression coefficients
with open(os.path.join(PHASE3_PATH, 'block_c_objects.pkl'), 'rb') as f:
    c_obj = pickle.load(f)
coef_df = c_obj['coef_df']

print("  Metrics loaded.\n")

# ── Derive summary statistics ─────────────────────────────────
overall_rate   = si['dropout'].mean() * 100
n_total        = len(si)
n_dropout      = si['dropout'].sum()
n_features     = len(FEATURES)

best_auc       = metrics_df.loc[best_model_name, 'ROC-AUC']
best_recall    = metrics_df.loc[best_model_name, 'Recall']
best_precision = metrics_df.loc[best_model_name, 'Precision']
best_f1        = metrics_df.loc[best_model_name, 'F1']

tier_counts    = risk_table['risk_tier'].value_counts()
n_high         = tier_counts.get('HIGH',   0)
n_medium       = tier_counts.get('MEDIUM', 0)
n_low          = tier_counts.get('LOW',    0)

# Top risk factors from Phase 3 logistic regression
feat_rows = coef_df[coef_df['Feature'] != 'Intercept'].copy()
feat_rows['OR'] = np.exp(feat_rows['Coef (β)'])
top_risk_feat  = feat_rows.loc[feat_rows[feat_rows['OR'] > 1]['OR'].idxmax(), 'Feature']
top_prot_feat  = feat_rows.loc[feat_rows[feat_rows['OR'] < 1]['OR'].idxmin(), 'Feature']

TODAY = date.today().strftime("%B %d, %Y")


# =============================================================
# BUILD THE EXECUTIVE SUMMARY
# =============================================================

DIVIDER = "=" * 70
SUBDIV  = "─" * 70
lines   = []

def L(text=""):
    lines.append(text)
    print(text)

L(DIVIDER)
L("STUDENT DROPOUT EARLY WARNING SYSTEM")
L("Executive Summary")
L(f"Open University Learning Analytics Dataset (OULAD)")
L(f"Prepared: {TODAY}")
L(DIVIDER)

# ── 1. Problem Statement ──────────────────────────────────────
L()
L("1. PROBLEM STATEMENT")
L(SUBDIV)
L("""
  Student dropout is one of the most costly challenges facing higher
  education institutions. When a student withdraws mid-course, the
  institution loses tuition revenue, the student loses time and
  confidence, and society loses a trained graduate.

  Early identification of at-risk students — before they withdraw —
  creates a window for targeted intervention: advisor outreach,
  additional support, or adjusted workload. The earlier the
  identification, the higher the chance of retention.

  This project asks: can we predict which students are most likely
  to drop out, early enough to intervene, and explain why?
""")

# ── 2. Data and Methodology ───────────────────────────────────
L("2. DATA AND METHODOLOGY")
L(SUBDIV)
L(f"""
  Dataset:
    Open University Learning Analytics Dataset (OULAD)
    {n_total:,} student-course enrolments across 7 course modules
    Real anonymised data from the UK Open University (2013–2014)

  Methodology — 5-phase end-to-end pipeline:
    Phase 1 — Data acquisition and quality checks
    Phase 2 — Exploratory data analysis (EDA)
    Phase 3 — Statistical analysis (chi-square, t-tests, logistic regression)
    Phase 4 — Machine learning model (Random Forest, XGBoost, SMOTE, SHAP)
    Phase 5 — Insights report and recommendations  ← this document

  Features used ({n_features} total):
    • Demographic    : gender, age band, disability status
    • Socioeconomic  : IMD deprivation band, prior education level
    • Academic       : previous attempts, studied credits
    • Behavioural    : assessment scores, % submissions, submission timing
    • Interaction    : education × deprivation, attempts × score

  Target variable:
    Binary dropout flag — 1 if student Withdrew, 0 otherwise
    Overall dropout rate: {overall_rate:.1f}%  ({n_dropout:,} of {n_total:,} students)
""")

# ── 3. Key Findings ───────────────────────────────────────────
L("3. KEY FINDINGS")
L(SUBDIV)
L("""
  Finding 1 — Dropout is widespread and unevenly distributed
    30%+ of enrolled students withdraw before completing their course.
    Dropout rates vary significantly across course modules, age groups,
    and education levels — suggesting structural and support-related
    causes, not just individual student factors.

  Finding 2 — Prior education is the strongest demographic predictor
    Students with no formal qualifications drop out at nearly twice
    the rate of postgraduate-educated students. This persists after
    controlling for all other variables in the logistic regression.

  Finding 3 — Socioeconomic deprivation compounds dropout risk
    Students from the most deprived areas (IMD 0–10%) show
    significantly higher dropout rates than those from affluent areas.
    The education × deprivation interaction reveals that low-educated
    students from deprived areas face a compounded disadvantage.

  Finding 4 — Assessment behaviour is the most powerful early signal
    Mean assessment score, % of assessments submitted, and first
    assessment score are the strongest predictors of dropout —
    stronger than any demographic variable. Crucially, this
    information is available within the first 4–6 weeks of term.

  Finding 5 — Most dropouts happen early in the course
    50% of all dropouts withdraw within the first half of the course.
    A model that scores students before the midpoint could flag the
    majority of at-risk students while there is still time to act.

  Finding 6 — Late registration is a dropout signal
    Students who register after the course start date have higher
    dropout rates, likely reflecting lower commitment or preparation.
""")

# ── 4. Model Performance ──────────────────────────────────────
L("4. MODEL PERFORMANCE")
L(SUBDIV)
L(f"""
  Four models were trained and compared:
    Logistic Regression  —  interpretable baseline
    Decision Tree        —  simple, fully explainable
    Random Forest        —  robust ensemble (tuned)
    XGBoost              —  gradient boosting (tuned)

  Best model: {best_model_name}

  Performance on held-out test set:
    ROC-AUC   : {best_auc:.3f}   (1.0 = perfect, 0.5 = random)
    Recall    : {best_recall:.3f}   (proportion of actual dropouts caught)
    Precision : {best_precision:.3f}   (proportion of flagged students who withdrew)
    F1-score  : {best_f1:.3f}   (harmonic mean of precision and recall)

  Interpretation:
    The model correctly identifies {best_recall*100:.1f}% of students who will
    drop out — meaning it catches {best_recall*100:.1f} out of every 100 real
    dropouts before they withdraw. The {(1-best_recall)*100:.1f}% it misses
    represents students whose early behaviour appears low-risk.

  Class imbalance handling:
    SMOTE oversampling was applied to the training set only,
    generating synthetic minority-class samples to balance the
    30/70 dropout/non-dropout split. The test set was never
    modified, ensuring honest evaluation.

  Explainability:
    SHAP (SHapley Additive exPlanations) values were computed for
    every student, enabling feature-level explanations for each
    individual prediction — not just global importance rankings.
""")

# ── 5. Risk Scoring Output ────────────────────────────────────
L("5. RISK SCORING OUTPUT")
L(SUBDIV)
L(f"""
  Every student was scored and assigned to a risk tier:

    🔴 HIGH risk   (dropout probability ≥ 0.70):  {n_high:,}  students  ({n_high/len(risk_table)*100:.1f}%)
    🟡 MEDIUM risk (probability 0.40 – 0.69):     {n_medium:,}  students  ({n_medium/len(risk_table)*100:.1f}%)
    🟢 LOW risk    (probability < 0.40):           {n_low:,}  students  ({n_low/len(risk_table)*100:.1f}%)

  The output is a ranked table of all {len(risk_table):,} students with:
    • Dropout probability (0–1 score)
    • Risk tier (HIGH / MEDIUM / LOW)
    • Key risk factors per student (via SHAP)
    • Demographic and course context

  Institutions can use this table to:
    • Prioritise which students advisors contact first
    • Tailor interventions to each student's specific risk factors
    • Monitor weekly score changes as new assessment data arrives
""")

# ── 6. Recommendations Preview ────────────────────────────────
L("6. RECOMMENDATIONS PREVIEW")
L(SUBDIV)
L("""
  Full recommendations are detailed in Block B of this report.
  In summary:

  For institutions:
    • Implement a weekly risk-scoring run using this model
    • Assign dedicated advisors to all HIGH-risk students by week 4
    • Create targeted support pathways for students with no formal
      qualifications entering credit-heavy modules

  For educators:
    • Use first-assessment scores as an early alert trigger
    • Investigate modules with above-average dropout rates for
      course design issues (pacing, assessment load, clarity)

  For policy:
    • Address the socioeconomic gap: students from the most deprived
      areas need additional financial and academic support structures
    • Track late-registration students as a high-priority cohort
""")

# ── 7. Limitations ────────────────────────────────────────────
L("7. LIMITATIONS AND NEXT STEPS")
L(SUBDIV)
L("""
  Limitations:
    • Dataset covers 2013–2014 OU data — student behaviour and
      technology access may have changed significantly since.
    • The model predicts withdrawal but cannot distinguish voluntary
      withdrawal (life circumstances) from academic withdrawal
      (struggling with content) — interventions differ.
    • SMOTE-generated synthetic samples may not reflect real
      student profiles at the extremes of the feature distribution.
    • Feature availability timing: some features (mean_score) are
      only available after assessments are submitted. A real-time
      system must track which features are available at each
      point in the course calendar.

  Recommended next steps:
    • Retrain on more recent data (post-2020 remote-learning cohorts)
    • Build a real-time scoring pipeline triggered by each new
      assessment submission
    • A/B test interventions: randomly assign at-risk students to
      intervention vs control groups to measure causal impact
    • Extend to multi-class prediction: distinguish Withdrawn /
      Fail / Pass / Distinction rather than binary dropout
    • Integrate student consent and data governance framework
      before deployment in a live institution
""")

L(DIVIDER)
L("END OF EXECUTIVE SUMMARY")
L(f"Full technical report: phases 1–4 notebooks and scripts")
L(f"Contact: [Your Name]  |  {TODAY}")
L(DIVIDER)


# =============================================================
# SAVE
# =============================================================
report_text = "\n".join(lines)
save_path   = os.path.join(OUTPUT_PATH, "phase5_executive_summary.txt")
with open(save_path, "w", encoding="utf-8") as f:
    f.write(report_text)

print(f"\n  ✅ Executive summary saved → {save_path}")
print(f"""
{'='*55}
BLOCK A COMPLETE — Executive Summary
{'='*55}

  Output: {save_path}

─────────────────────────────────────────────────────────
Next → Run phase5_block_b_recommendations.py
─────────────────────────────────────────────────────────
""")

Loading metrics from all phases...
  Metrics loaded.

STUDENT DROPOUT EARLY WARNING SYSTEM
Executive Summary
Open University Learning Analytics Dataset (OULAD)
Prepared: June 02, 2026

1. PROBLEM STATEMENT
──────────────────────────────────────────────────────────────────────

  Student dropout is one of the most costly challenges facing higher
  education institutions. When a student withdraws mid-course, the
  institution loses tuition revenue, the student loses time and
  confidence, and society loses a trained graduate.

  Early identification of at-risk students — before they withdraw —
  creates a window for targeted intervention: advisor outreach,
  additional support, or adjusted workload. The earlier the
  identification, the higher the chance of retention.

  This project asks: can we predict which students are most likely
  to drop out, early enough to intervene, and explain why?

2. DATA AND METHODOLOGY
──────────────────────────────────────────────────────────────────────
